In [ ]:
import os
import pandas as pd
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2
%matplotlib inline

In [ ]:
def plot_multiple(image_paths, image_labels=None,nrow=1,figsize=None):
    ncol = int(np.ceil(len(image_paths)/nrow))
    
    if figsize is None:
        figsize=(ncol, nrow)
        
    print(ncol)
    fig, ax = plt.subplots(nrow,ncol, figsize=figsize)

    for id_image, image_path in enumerate(image_paths):
        if os.path.exists(image_path):
            img_now = cv2.cvtColor(cv2.imread(image_path),cv2.COLOR_BGR2RGB) 
        else:
            img_now = image_path

        if nrow>1:
            ax[id_image%nrow,id_image//nrow].imshow(img_now)
            ax[id_image%nrow,id_image//nrow].axis("off")
            if image_labels is not None:
                ax[id_image%nrow,id_image//nrow].set_title(image_labels[id_image])
        else:
            ax[id_image].imshow(img_now)
            ax[id_image].axis("off")
            if image_labels is not None:
                ax[id_image].set_title(image_labels[id_image])
   
    return fig,ax

In [ ]:
dfd_100_path = "/mnt/ssd/workspace/adi/vh_repos_byversion/deepfake-detection/1-0-0/face-production-face-deepfake/evaluation_tools/test_triton-notriton_fd3-1-0_output.txt"
dfd_120_path = "/mnt/ssd/workspace/adi/repos/vh_deepfake_trainer/tools/evaluation_tools/tools/test_multimodel_prod_prod-data.txt"

In [ ]:
pd_dfd_100 = pd.read_csv(dfd_100_path,sep='|')
pd_dfd_120 = pd.read_csv(dfd_120_path,sep='|')

In [ ]:
pd_dfd_100.head()

In [ ]:
pd_dfd_120.head()

In [ ]:
pd_join = pd_dfd_100.merge(pd_dfd_120[['image_name','pred_label','deepfake_score']],on='image_name',suffixes=("_1-0-0","_1-2-0"))

In [ ]:
pd_join["original_pred"] = pd_join["image_name"].str.split('/').apply(lambda x:x[9])

In [ ]:
pd_join["pred_1-0-0"] = pd_join["deepfake_score_1-0-0"].apply(lambda x:"DEEPFAKE" if x>0.75 else "REAL")
pd_join["pred_1-2-0_0,5"] = pd_join["deepfake_score_1-2-0"].apply(lambda x:"DEEPFAKE" if x>0.5 else "REAL")
pd_join["pred_1-2-0_0,6"] = pd_join["deepfake_score_1-2-0"].apply(lambda x:"DEEPFAKE" if x>0.6 else "REAL")
pd_join["pred_1-2-0_0,7"] = pd_join["deepfake_score_1-2-0"].apply(lambda x:"DEEPFAKE" if x>0.7 else "REAL")
pd_join["pred_1-2-0_0,75"] = pd_join["deepfake_score_1-2-0"].apply(lambda x:"DEEPFAKE" if x>0.75 else "REAL")

In [ ]:
pd_join.groupby(["pred_1-0-0","pred_1-2-0_0,5"]).count()["image_name"]

In [ ]:
grouped_data = pd_join.groupby(["pred_1-0-0"]).count()[["image_name"]]
grouped_data['percentage'] = grouped_data.apply(lambda x: 100 * x / x.sum())
grouped_data

In [ ]:
grouped_data = pd_join.groupby(["pred_1-2-0_0,5"]).count()[["image_name"]]
grouped_data['percentage'] = grouped_data.apply(lambda x: 100 * x / x.sum())
grouped_data

grouped_data = pd_join.groupby(["pred_1-2-0_0,6"]).count()[["image_name"]]
grouped_data['percentage'] = grouped_data.apply(lambda x: 100 * x / x.sum())
grouped_data

grouped_data = pd_join.groupby(["pred_1-2-0_0,75"]).count()[["image_name"]]
grouped_data['percentage'] = grouped_data.apply(lambda x: 100 * x / x.sum())
grouped_data

In [ ]:
df_select = pd_join[(pd_join["pred_1-0-0"]!=pd_join["pred_1-2-0_0,75"]) & (pd_join["pred_1-0-0"]=="REAL")]
df_plot = df_select.iloc[:30]

list_label = []
for index, row in df_plot.iterrows():
    list_label.append(f"DFD 1.0.0:{round(row['deepfake_score_1-0-0'],2)} \nDFD 1.2.0:{round(row['deepfake_score_1-2-0'],2)}")
fig,ax = plot_multiple(df_plot["image_name"], image_labels=list_label,nrow=3,figsize=(20,8))
#np.round(df_plot["deepfake_score_1-2-0"].tolist(),2)

In [ ]:
df_select = pd_join[(pd_join["pred_1-0-0"]!=pd_join["pred_1-2-0_0,75"]) & (pd_join["pred_1-0-0"]=="DEEPFAKE")]
df_plot = df_select.iloc[:30]

list_label = []
for index, row in df_plot.iterrows():
    list_label.append(f"DFD 1.0.0:{round(row['deepfake_score_1-0-0'],2)} \nDFD 1.2.0:{round(row['deepfake_score_1-2-0'],2)}")
fig,ax = plot_multiple(df_plot["image_name"], image_labels=list_label,nrow=3,figsize=(20,8))
#np.round(df_plot["deepfake_score_1-2-0"].tolist(),2)

In [ ]:
df_select = pd_join[(pd_join["pred_1-0-0"]==pd_join["pred_1-2-0_0,75"]) & (pd_join["pred_1-0-0"]=="DEEPFAKE")]
df_plot = df_select.iloc[:30]

list_label = []
for index, row in df_plot.iterrows():
    list_label.append(f"DFD 1.0.0:{round(row['deepfake_score_1-0-0'],2)} \nDFD 1.2.0:{round(row['deepfake_score_1-2-0'],2)}")
fig,ax = plot_multiple(df_plot["image_name"], image_labels=list_label,nrow=3,figsize=(20,8))
#np.round(df_plot["deepfake_score_1-2-0"].tolist(),2)

In [ ]:
df_select = pd_join[(pd_join["deepfake_score_1-2-0"]>0.6)&(pd_join["deepfake_score_1-2-0"]<=0.75)]
df_plot = df_select.iloc[:30]

list_label = []
for index, row in df_plot.iterrows():
    list_label.append(f"DFD 1.0.0:{round(row['deepfake_score_1-0-0'],2)} \nDFD 1.2.0:{round(row['deepfake_score_1-2-0'],2)}")
fig,ax = plot_multiple(df_plot["image_name"], image_labels=list_label,nrow=3,figsize=(20,8))
#np.round(df_plot["deepfake_score_1-2-0"].tolist(),2)

In [ ]:
import shutil
#shutil.copy2(source_file, destination_file)

In [ ]:
from PIL import Image, ImageDraw, ImageFont

def add_text_to_image(input_path, output_path, text):
    try:
        with Image.open(input_path) as img:
            draw = ImageDraw.Draw(img)

            # --- 1. Calculate Font Size (5% of image height) ---
            # We convert to int() because font size must be a whole number
            font_size = int(img.height * 0.05)
            
            # Ensure a minimum size so it doesn't disappear on tiny images
            font_size = max(font_size, 10)

            try:
                # Try loading Arial (common on Windows)
                font = ImageFont.truetype("arial.ttf", font_size)
            except IOError:
                try:
                    # Fallback for Mac/Linux (DejaVu is common)
                    font = ImageFont.truetype("DejaVuSans.ttf", font_size)
                except IOError:
                    print("Could not find a scalable font. Falling back to default (size will be fixed).")
                    font = ImageFont.load_default()

            # 4. Define text position (x, y)
            # (0, 0) is the top-left corner of the image
            x = 20
            y = int(img.height * 0.05)

            # 5. Define text color (R, G, B)
            text_color = (255, 255, 255) # White
            
            # Optional: Add a black outline/stroke for better visibility
            outline_color = (0, 0, 0)

            # 6. Draw the text
            draw.text(
                (x,y), 
                text, 
                fill=text_color, 
                font=font, 
                stroke_width=2, 
                stroke_fill=outline_color
            )

            # 7. Save the result
            img.save(output_path)
            print(f"Success! Image saved to {output_path}")

    except Exception as e:
        print(f"An error occurred: {e}")
        # 1. Open the image
        with Image.open(input_path) as img:
            # 2. Create a drawing context
            draw = ImageDraw.Draw(img)

            # 3. Load a font
            # On Windows, you might use: "arial.ttf"
            # On Mac/Linux, you might use: "Arial.ttf" or a full path
            # If the font isn't found, it falls back to a default simple font
            try:
                # Size 50 font
                font = ImageFont.truetype("arial.ttf", 50) 
            except IOError:
                font = ImageFont.load_default()
                print("Custom font not found. Using default font.")

            # 4. Define text position (x, y)
            # (0, 0) is the top-left corner of the image
            position = (50, 50)

            # 5. Define text color (R, G, B)
            text_color = (255, 255, 255) # White
            
            # Optional: Add a black outline/stroke for better visibility
            outline_color = (0, 0, 0)

            # 6. Draw the text
            draw.text(
                position, 
                text, 
                fill=text_color, 
                font=font, 
                stroke_width=2, 
                stroke_fill=outline_color
            )

            # 7. Save the result
            img.save(output_path)
            print(f"Success! Image saved to {output_path}")

In [ ]:
for idx_row,row_now in df_select.iloc[:200].iterrows():
    image_name = row_now["image_name"].split('/')[-1]
    tgt_folder = "/mnt/ssd/workspace/adi/repos/vh_deepfake_trainer/tools/evaluation_tools/tools/saved_samples"
    text = f"DFD-1.2.0 score: {round(row_now['deepfake_score_1-2-0'],3)}"
    add_text_to_image(row_now["image_name"], os.path.join(tgt_folder,image_name), text)
    #shutil.copy2(img_now, os.path.join(tgt_folder,image_name))

In [ ]:
img_now

In [ ]:
os.path.join(tgt_folder,image_name)

In [ ]:
df_select = pd_join[pd_join["deepfake_score_1-2-0"]>0.75]
df_plot = df_select.iloc[:30]

list_label = []
for index, row in df_plot.iterrows():
    list_label.append(f"DFD 1.0.0:{round(row['deepfake_score_1-0-0'],2)} \nDFD 1.2.0:{round(row['deepfake_score_1-2-0'],2)}")
fig,ax = plot_multiple(df_plot["image_name"], image_labels=list_label,nrow=3,figsize=(20,8))
#np.round(df_plot["deepfake_score_1-2-0"].tolist(),2)

In [ ]:
df_select = pd_join[pd_join["deepfake_score_1-2-0"]>0.75]
df_plot = df_select.iloc[:30]